2026년 8월 23일 기준으로 pyannote/speaker-diarization-3.1 모델을 구동하기 위한 핵심 라이브러리인 pyannote.audio 및 관련 패키지의 NumPy 버전 요구사항은 아래 리포지토리에서 확인할 수 있다.

https://github.com/pyannote/pyannote-core/blob/develop/pyproject.toml

In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

HUGGING_FACE_TOKEN = os.getenv("HUGGING_FACE_TOKEN")

In [7]:
# instantiate the pipeline
from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained(
  "pyannote/speaker-diarization-3.1",
  use_auth_token=HUGGING_FACE_TOKEN
)

In [8]:
import torch

# cuda가 사용 가능한 경우 cuda를 사용하도록 설정
if torch.cuda.is_available():
    pipeline.to(torch.device("cuda"))
    print('cuda is available')
else:
    print('cuda is not available')

cuda is available


In [9]:
# run the pipeline on an audio file
# diarization = pipeline("audio.wav")
diarization = pipeline("audio/싼기타_비싼기타.mp3")

# dump the diarization output to disk using RTTM format
with open("싼기타_비싼기타.rttm", "w", encoding='utf-8') as rttm:
    diarization.write_rttm(rttm)

c:\Users\Unreon\Documents\Git\STUDY\Sungmin\chap05\.venv\Lib\site-packages\torchaudio\_backend\utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
c:\Users\Unreon\Documents\Git\STUDY\Sungmin\chap05\.venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)
c:\Users\Unre

In [10]:
# RTTM을 CSV로 변환
import pandas as pd
rttm_path = "싼기타_비싼기타.rttm"

df_rttm = pd.read_csv(
    rttm_path,      # rttm 파일 경로
    sep=' ',        # 구분자는 띄어쓰기
    header=None,    # 헤더는 없음
    names=['type', 'file', 'chnl', 'start', 'duration', 'C1', 'C2', 'speaker_id', 'C3', 'C4'] 
)

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4
0,SPEAKER,싼기타_비싼기타,1,0.976,5.823,NaN,NaN,SPEAKER_01,NaN,NaN
1,SPEAKER,싼기타_비싼기타,1,7.394,3.990,NaN,NaN,SPEAKER_01,NaN,NaN
2,SPEAKER,싼기타_비싼기타,1,11.740,4.941,NaN,NaN,SPEAKER_01,NaN,NaN
3,SPEAKER,싼기타_비싼기타,1,17.224,10.662,NaN,NaN,SPEAKER_01,NaN,NaN
4,SPEAKER,싼기타_비싼기타,1,28.667,1.528,NaN,NaN,SPEAKER_01,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
83,SPEAKER,싼기타_비싼기타,1,414.474,2.971,NaN,NaN,SPEAKER_00,NaN,NaN
84,SPEAKER,싼기타_비싼기타,1,417.750,3.480,NaN,NaN,SPEAKER_01,NaN,NaN
85,SPEAKER,싼기타_비싼기타,1,423.642,0.781,NaN,NaN,SPEAKER_00,NaN,NaN
86,SPEAKER,싼기타_비싼기타,1,424.745,3.531,NaN,NaN,SPEAKER_00,NaN,NaN


In [11]:
# start + duration을 end로 변환
df_rttm['end'] = df_rttm['start'] + df_rttm['duration']

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end
0,SPEAKER,싼기타_비싼기타,1,0.976,5.823,NaN,NaN,SPEAKER_01,NaN,NaN,6.799
1,SPEAKER,싼기타_비싼기타,1,7.394,3.990,NaN,NaN,SPEAKER_01,NaN,NaN,11.384
2,SPEAKER,싼기타_비싼기타,1,11.740,4.941,NaN,NaN,SPEAKER_01,NaN,NaN,16.681
3,SPEAKER,싼기타_비싼기타,1,17.224,10.662,NaN,NaN,SPEAKER_01,NaN,NaN,27.886
4,SPEAKER,싼기타_비싼기타,1,28.667,1.528,NaN,NaN,SPEAKER_01,NaN,NaN,30.195
...,...,...,...,...,...,...,...,...,...,...,...
83,SPEAKER,싼기타_비싼기타,1,414.474,2.971,NaN,NaN,SPEAKER_00,NaN,NaN,417.445
84,SPEAKER,싼기타_비싼기타,1,417.750,3.480,NaN,NaN,SPEAKER_01,NaN,NaN,421.230
85,SPEAKER,싼기타_비싼기타,1,423.642,0.781,NaN,NaN,SPEAKER_00,NaN,NaN,424.423
86,SPEAKER,싼기타_비싼기타,1,424.745,3.531,NaN,NaN,SPEAKER_00,NaN,NaN,428.276


In [12]:
df_rttm["number"] = None	# number 열 만들고 None으로 초기화
df_rttm.at[0, "number"] = 0

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.976,5.823,NaN,NaN,SPEAKER_01,NaN,NaN,6.799,0
1,SPEAKER,싼기타_비싼기타,1,7.394,3.990,NaN,NaN,SPEAKER_01,NaN,NaN,11.384,None
2,SPEAKER,싼기타_비싼기타,1,11.740,4.941,NaN,NaN,SPEAKER_01,NaN,NaN,16.681,None
3,SPEAKER,싼기타_비싼기타,1,17.224,10.662,NaN,NaN,SPEAKER_01,NaN,NaN,27.886,None
4,SPEAKER,싼기타_비싼기타,1,28.667,1.528,NaN,NaN,SPEAKER_01,NaN,NaN,30.195,None
...,...,...,...,...,...,...,...,...,...,...,...,...
83,SPEAKER,싼기타_비싼기타,1,414.474,2.971,NaN,NaN,SPEAKER_00,NaN,NaN,417.445,None
84,SPEAKER,싼기타_비싼기타,1,417.750,3.480,NaN,NaN,SPEAKER_01,NaN,NaN,421.230,None
85,SPEAKER,싼기타_비싼기타,1,423.642,0.781,NaN,NaN,SPEAKER_00,NaN,NaN,424.423,None
86,SPEAKER,싼기타_비싼기타,1,424.745,3.531,NaN,NaN,SPEAKER_00,NaN,NaN,428.276,None


In [13]:
for i in range(1, len(df_rttm)):
    if df_rttm.at[i, "speaker_id"] != df_rttm.at[i-1, "speaker_id"]:
        df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"] + 1
    else:
        df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"]

display(df_rttm.head(10)) 

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.976,5.823,NaN,NaN,SPEAKER_01,NaN,NaN,6.799,0
1,SPEAKER,싼기타_비싼기타,1,7.394,3.990,NaN,NaN,SPEAKER_01,NaN,NaN,11.384,0
2,SPEAKER,싼기타_비싼기타,1,11.740,4.941,NaN,NaN,SPEAKER_01,NaN,NaN,16.681,0
3,SPEAKER,싼기타_비싼기타,1,17.224,10.662,NaN,NaN,SPEAKER_01,NaN,NaN,27.886,0
4,SPEAKER,싼기타_비싼기타,1,28.667,1.528,NaN,NaN,SPEAKER_01,NaN,NaN,30.195,0
5,SPEAKER,싼기타_비싼기타,1,32.419,0.764,NaN,NaN,SPEAKER_00,NaN,NaN,33.183,1
6,SPEAKER,싼기타_비싼기타,1,33.557,3.548,NaN,NaN,SPEAKER_00,NaN,NaN,37.105,1
7,SPEAKER,싼기타_비싼기타,1,37.632,3.769,NaN,NaN,SPEAKER_00,NaN,NaN,41.401,1
8,SPEAKER,싼기타_비싼기타,1,41.621,0.017,NaN,NaN,SPEAKER_00,NaN,NaN,41.638,1
9,SPEAKER,싼기타_비싼기타,1,41.638,0.815,NaN,NaN,SPEAKER_01,NaN,NaN,42.453,2


In [14]:
df_rttm_grouped = df_rttm.groupby("number").agg(
    start=pd.NamedAgg(column='start', aggfunc='min'),
    end=pd.NamedAgg(column='end', aggfunc='max'),
    speaker_id=pd.NamedAgg(column='speaker_id', aggfunc='first')
)

display(df_rttm_grouped)

,start,end,speaker_id
number,,,
0,0.976,30.195,SPEAKER_01
1,32.419,41.638,SPEAKER_00
2,41.638,42.453,SPEAKER_01
3,41.655,42.725,SPEAKER_00
4,42.674,44.032,SPEAKER_01
5,45.798,67.105,SPEAKER_00
6,67.224,82.793,SPEAKER_01
7,84.643,102.572,SPEAKER_00
8,103.489,117.530,SPEAKER_01


In [15]:
df_rttm_grouped["duration"] = df_rttm_grouped["end"] - df_rttm_grouped["start"]
df_rttm_grouped = df_rttm_grouped.reset_index(drop=True)
display(df_rttm_grouped)

,start,end,speaker_id,duration
0,0.976,30.195,SPEAKER_01,29.219
1,32.419,41.638,SPEAKER_00,9.219
2,41.638,42.453,SPEAKER_01,0.815
3,41.655,42.725,SPEAKER_00,1.070
4,42.674,44.032,SPEAKER_01,1.358
5,45.798,67.105,SPEAKER_00,21.307
6,67.224,82.793,SPEAKER_01,15.569
7,84.643,102.572,SPEAKER_00,17.929
8,103.489,117.530,SPEAKER_01,14.041
9,119.737,138.667,SPEAKER_00,18.930


In [16]:
df_rttm_grouped.to_csv(
    "audio/싼기타_비싼기타_rttm.csv",
    sep=',',
    index=False
)